Bibliotecas Necessárias

In [1]:
import json

from bs4 import BeautifulSoup

import requests

from google import genai

import os

from dotenv import load_dotenv

import smtplib

import time

Acessando a chave da API através das variáveis de ambiente

In [2]:
load_dotenv()

CHAVE_API = os.getenv('CHAVE_API')

Instanciando a classe genai

In [3]:
gemini = genai.Client(api_key=CHAVE_API)

lendo o arquivo de endereços

In [14]:
caminho_enderecos = 'produtos.json'

with open(caminho_enderecos, "r") as enderecos:
    
    dicionario_enderecos = json.load(enderecos)

dicionario_enderecos
    

{'https://produto.mercadolivre.com.br/MLB-5318215486-kit-4-pulseira-couro-masculina-marrom-regulavel-bolinha-punk-_JM#intervention_type=full&searchVariation=187220636913&backend_model=search-backend&position=4&search_layout=grid&type=cart_intervention&tracking_id=702f9417-a3c6-4f37-9677-f47e8d7ba9bc': {'loja': 'mercado livre',
  'produto': 'pulseira de couro',
  'preco': 25.0},
 'https://produto.mercadolivre.com.br/MLB-3848543993-camiseta-tradicional-de-algodo-banda-de-rock-sepultura-_JM?searchVariation=184841542449#polycard_client=search-desktop&be_origin=backend&searchVariation=184841542449&search_layout=grid&position=6&type=item&tracking_id=ce5441d6-9c84-4ec9-81ba-1d83f0f9ec8e&sid=search': {'loja': 'mercado livre',
  'produto': 'camiseta do sepultura',
  'preco': 45.0},
 'https://www.netshoes.com.br/p/tenis-converse-chuck-taylor-all-star-cano-baixo-D26-0147-002': {'loja': 'netshoes',
  'produto': 'all stars',
  'preco': 230.0}}

Inserindo links no sistema (Este trecho só é obrigatório no primeiro acesso)

In [15]:
link = input("Link da página que será acessada")

nome_loja = input("Nome da loja")

nome_produto = input("Nome do produto: ")

faixa_preco = input("Preço alvo (avisar se o valor for menor ou igual a):")


if link != '' and nome_loja != '' and faixa_preco != '':
    
    faixa_preco = faixa_preco.replace(",", ".")

    faixa_preco = float(faixa_preco)
        
    dicionario_enderecos[link] = {"loja": nome_loja, "produto": nome_produto,"preco":faixa_preco}
        
    with open(caminho_enderecos, "w") as arquivo:
        
        json.dump(dicionario_enderecos, arquivo, indent=4)

else:
    
    print("Por favor preencha todos os campos")


dicionario_enderecos

{'https://produto.mercadolivre.com.br/MLB-5318215486-kit-4-pulseira-couro-masculina-marrom-regulavel-bolinha-punk-_JM#intervention_type=full&searchVariation=187220636913&backend_model=search-backend&position=4&search_layout=grid&type=cart_intervention&tracking_id=702f9417-a3c6-4f37-9677-f47e8d7ba9bc': {'loja': 'mercado livre',
  'produto': 'pulseira de couro',
  'preco': 25.0},
 'https://produto.mercadolivre.com.br/MLB-3848543993-camiseta-tradicional-de-algodo-banda-de-rock-sepultura-_JM?searchVariation=184841542449#polycard_client=search-desktop&be_origin=backend&searchVariation=184841542449&search_layout=grid&position=6&type=item&tracking_id=ce5441d6-9c84-4ec9-81ba-1d83f0f9ec8e&sid=search': {'loja': 'mercado livre',
  'produto': 'camiseta do sepultura',
  'preco': 45.0},
 'https://www.netshoes.com.br/p/tenis-converse-chuck-taylor-all-star-cano-baixo-D26-0147-002': {'loja': 'netshoes',
  'produto': 'all stars',
  'preco': 230.0},
 'https://www.netshoes.com.br/p/jaqueta-corta-vento-res

Acessando a lista de sites

In [17]:
for url, dados_produtos in dicionario_enderecos.items():
    
    time.sleep(10)
    
    headers = {"User-Agent": "Chrome/149.0.7827.103",
               "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
               "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
               "Accept-Encoding": "gzip, deflate, br",
               "Connection": "keep-alive",
               "Upgrade-Insecure-Requests": "1",
               "Sec-Fetch-Dest": "document",
               "Sec-Fetch-Mode": "navigate",
               "Sec-Fetch-Site": "none",
               "Sec-Fetch-User": "?1"}
    
    requisicao = requests.get(url, headers=headers)
    
    if requisicao.status_code != 200:
        
        print(f"Não foi possivel acessar o site {dados_produtos['loja']}. Status: {requisicao.status_code}\n")
    
    else:
        
        html = requisicao.text
        
        sopa = BeautifulSoup(html, "html.parser")
        
        try:
            
        
            resposta = gemini.models.generate_content(
            
            model = 'gemini-2.5-flash',
            contents = f"Análise o HTML de cada url retornada e colete o preço de cada produto. HTML dos e-commerce: {sopa}. Por favor, caso você encontre o preço do produto no html, retorne apenas o valor do produto (sem utilizar qualquer outro texto, apenas o valor bruto), pois irei usar esse valor na automação que estou construindo"
            )
            
            
            if "ERRO" in resposta.text:
                
                print(f"O sistema não encontrou o preço do {dados_produtos['produto']} no site {dados_produtos['loja']}\n")
                
                print(f"{resposta.text}\n")
            
            else:
            
                preco_atual_texto = resposta.text.replace(",",".")
                
                preco_atual = float(preco_atual_texto)
                
                print(f"Preço atual: {preco_atual}")
                
                if preco_atual <= dados_produtos['preco']:
                    
                    print(f"A loja {dados_produtos['loja']} esta com produtos abaixo de {dados_produtos['preco']}")
                    
        except Exception as erro:
            
            print(f"A API do gemini retornou um erro: {erro}")
        

A API do gemini retornou um erro: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 55.300641046s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model':